# import libraries and data

In [1]:
import numpy as np 
import pandas as pd 
import json
from matplotlib import pyplot as plt
from pathlib import Path
import os

In [2]:
track1 = pd.read_json('/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track1_train.json')
track2 = pd.read_json('/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track2_train.json')
global track1 
global track2

In [3]:
track1.head()

,patient_id,left,right
0,1,"{'1': 0, '2': 1, '3': 0, '4': 1, '5': 1, '6': ...","{'1': 0, '2': 1, '3': 0, '4': 1, '5': 1, '6': ..."
1,2,"{'1': 0, '2': 0, '3': 0, '4': 0, '5': 0, '6': ...","{'1': 1, '2': 1, '3': 0, '4': 0, '5': 1, '6': ..."
2,3,"{'1': 0, '2': 1, '3': 0, '4': 0, '5': 0, '6': ...","{'1': 1, '2': 1, '3': 0, '4': 0, '5': 1, '6': ..."
3,6,"{'1': 1, '2': 1, '3': 0, '4': 1, '5': 0, '6': ...","{'1': 0, '2': 0, '3': 0, '4': 1, '5': 0, '6': ..."
4,7,"{'1': 1, '2': 1, '3': 0, '4': 0, '5': 1, '6': ...","{'1': 1, '2': 1, '3': 0, '4': 0, '5': 1, '6': ..."


In [4]:
track2.head()

,patient_id,left,right
0,1,{'gait_subtype': 'type3'},{'gait_subtype': 'type3'}
1,5,{'gait_subtype': 'type3'},{'gait_subtype': 'type3'}
2,12,{'gait_subtype': 'type3'},{'gait_subtype': 'type3'}
3,18,{'gait_subtype': 'type2'},{'gait_subtype': 'type2'}
4,21,{'gait_subtype': 'type2'},{'gait_subtype': 'type2'}


# Metadata Preprocessing & EDA

In [5]:
def processing_metadata(track1 = track1, track2 = track2):
    for i in range(1, 18):
        track1['L'+str(i)] = track1['left'].apply(lambda x: x[str(i)])
        track1['R'+str(i)] = track1['right'].apply(lambda x: x[str(i)])
    track1_scored_item = ['L']
    track1 = track1.drop(['left', 'right'], axis=1)
    track2['Left_gait_subtype'] = track2['left'].map(lambda x: x['gait_subtype'])
    track2['Right_gait_subtype'] = track2['right'].map(lambda x: x['gait_subtype'])
    track2 = track2.drop(['left', 'right'], axis = 1)
    return track1, track2

In [6]:
track1, track2 = processing_metadata()
track1.head()

,patient_id,L1,R1,L2,R2,L3,R3,L4,R4,L5,...,L13,R13,L14,R14,L15,R15,L16,R16,L17,R17
0,1,0,0,1,1,0,0,1,1,1,...,0,0,1,0,1,1,1,1,0,0
1,2,0,1,0,1,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
2,3,0,1,1,1,0,0,0,0,0,...,0,0,0,1,1,0,1,1,0,1
3,6,1,0,1,0,0,0,1,1,0,...,0,0,0,0,1,1,0,1,0,0
4,7,1,1,1,1,0,0,0,0,1,...,0,0,1,0,1,1,1,1,1,1


In [7]:
track2.head()

,patient_id,Left_gait_subtype,Right_gait_subtype
0,1,type3,type3
1,5,type3,type3
2,12,type3,type3
3,18,type2,type2
4,21,type2,type2


# Q1: how many patients do we have

In [8]:
print(track1.shape[0])
print(track2.shape[0])

94
22


## Q2: do patients have the same gait type at left and right?

In [9]:
track2.loc[track2['Left_gait_subtype']!=track2['Right_gait_subtype']]

,patient_id,Left_gait_subtype,Right_gait_subtype
11,32,type2,type3
15,48,type1,type3


## Q3: how many patients have left EVGS scoring different from their right?

In [10]:
def sum_svgs_per_leg(track1 = track1):
    left = ['L'+str(i) for i in range(1, 18)]
    right = ['R'+str(i) for i in range(1, 18)]
    track1['Left_SVGS_Score'] = track1.loc[:,left].sum(axis = 1)
    track1['Right_SVGS_Score'] = track1.loc[:,right].sum(axis = 1)
    return track1

In [11]:
track1 = sum_svgs_per_leg()

In [12]:
track1[['Left_SVGS_Score', 'Right_SVGS_Score']].describe()

,Left_SVGS_Score,Right_SVGS_Score
count,94.000000,94.000000
mean,5.840426,5.989362
std,3.312742,3.227885
min,1.000000,0.000000
25%,3.000000,3.000000
50%,5.000000,6.000000
75%,8.000000,8.000000
max,13.000000,14.000000


In [13]:
def get_id(i):
    l = 4
    df = l-len(str(i))
    return '0'*df+str(i)

In [20]:
def organize_df(side):
    cols = ['patient_id','file', 'nose', 'shoulder_l', 'shoulder_r', 'hip_l', 'hip_r', 'knee_l', 'knee_r', 'ankle_l', 
             'ankle_r', 'toe1_l', 'toe1_r', 'toe5_l', 'toe5_r', 'heel_l', 'heel_r', 'nose_CI', 'shoulder_l_CI', 
             'shoulder_r_CI', 'hip_l_CI', 'hip_r_CI', 'knee_l_CI', 'knee_r_CI','ankle_l_CI', 'ankle_r_CI',
            'toe1_l_CI', 'toe1_r_CI', 'toe5_l_CI','toe5_r_CI', 'heel_l_CI', 'heel_r_CI', 'bbox']
    
    rows = [] 
    points2keep = [1, 6, 7, 12, 13, 14, 15, 16, 17, 18, 21, 19, 22, 20, 23]
    
    for i in range(110):
        patient_id = i + 1
        print(f'Processing patient: {patient_id}')
        path = f'/kaggle/input/datasets/adastroabyssosque/first-ai-for-children-competition-data/dataset/{get_id(patient_id)}'
        base_path = Path(path)
        
        # Only grab .json files to speed up the glob
        all_frames = sorted([
            f for f in base_path.rglob("*.json") 
            if side in f.parent.name.lower()
        ])
        
        for frame in all_frames:
            try:
                with open(frame, 'r') as f:
                    data = json.load(f)
                    inst = data['instance_info'][0]
                    
                    kp = inst['keypoints']
                    ks = inst['keypoint_scores']
                    bbox = inst.get('gt_bbox_xywh_px') # .get is safer than try/except
                    
                    # Create the record directly
                    record = [patient_id, str(frame)]
                    record.extend([kp[idx-1] for idx in points2keep])
                    record.extend([ks[idx-1] for idx in points2keep])
                    record.append(bbox)
                    
                    # Store as a dictionary for the final DF
                    rows.append(dict(zip(cols, record)))
                    
            except Exception:
                print(f"Error processing: {frame}")
                
    return pd.DataFrame(rows, columns = cols) # Create DF once at the end


In [23]:
print('Generating Dataset for Frontal View:')
front_df = organize_df('forward')
print('Generating Dataset for Backward View: ')
back_df = organize_df('backward')
print('Generating Dataset for Left View: ')
left_df = organize_df('left')
print('Generating Dataset for Right View: ')
right_df = organize_df('right')

Generating Dataset for Frontal View:
Processing patient: 1
Processing patient: 2
Processing patient: 3
Processing patient: 4
Processing patient: 5
Processing patient: 6
Processing patient: 7
Processing patient: 8
Processing patient: 9
Processing patient: 10
Processing patient: 11
Processing patient: 12
Processing patient: 13
Processing patient: 14
Processing patient: 15
Processing patient: 16
Processing patient: 17
Processing patient: 18
Processing patient: 19
Processing patient: 20
Processing patient: 21
Processing patient: 22
Processing patient: 23
Processing patient: 24
Processing patient: 25
Processing patient: 26
Processing patient: 27
Processing patient: 28
Processing patient: 29
Processing patient: 30
Processing patient: 31
Processing patient: 32
Processing patient: 33
Processing patient: 34
Processing patient: 35
Processing patient: 36
Processing patient: 37
Processing patient: 38
Processing patient: 39
Processing patient: 40
Processing patient: 41
Processing patient: 42
Proces

In [30]:
print(front_df.shape)
print(back_df.shape)
print(left_df.shape)
print(right_df.shape)

(90107, 33)
(83560, 33)
(88235, 33)
(77240, 33)


In [45]:
T1 = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2 = [4, 6, 7, 13, 26, 35, 39, 42, 50]

test_t1_front = front_df.loc[front_df['patient_id'].isin(T1)]
test_t1_back = back_df.loc[back_df['patient_id'].isin(T1)]
test_t1_left = left_df.loc[left_df['patient_id'].isin(T1)]
test_t1_right = right_df.loc[left_df['patient_id'].isin(T1)]
test_t2_front = front_df.loc[front_df['patient_id'].isin(T2)]
test_t2_back = back_df.loc[back_df['patient_id'].isin(T2)]
test_t2_left = left_df.loc[left_df['patient_id'].isin(T2)]
test_t2_right = right_df.loc[left_df['patient_id'].isin(T2)]

print('T1 data shape in test T1: ')
print(f'Front: {test_t1_front.shape}')
print(f'Back: {test_t1_back.shape}')
print(f'Left: {test_t1_left.shape}')
print(f'Right: {test_t1_right.shape}')
print('T2 data shape in test T1: ')
print(f'Front: {test_t2_front.shape}')
print(f'Back: {test_t2_back.shape}')
print(f'Left: {test_t2_left.shape}')
print(f'Right: {test_t2_right.shape}')

train_t1_front = front_df.merge(track1, on = 'patient_id', how = 'inner')
train_t1_back = back_df.merge(track1, on = 'patient_id', how = 'inner')
train_t1_left = left_df.merge(track1, on = 'patient_id', how = 'inner')
train_t1_right = right_df.merge(track1, on = 'patient_id', how = 'inner')
train_t2_front = front_df.merge(track2, on = 'patient_id', how = 'inner')
train_t2_back = back_df.merge(track2, on = 'patient_id', how = 'inner')
train_t2_left = left_df.merge(track2, on = 'patient_id', how = 'inner')
train_t2_right = right_df.merge(track2, on = 'patient_id', how = 'inner')

print('T1 data shape in train T1: ')
print(f'Front: {train_t1_front.shape}')
print(f'Back: {train_t1_back.shape}')
print(f'Left: {train_t1_left.shape}')
print(f'Right: {train_t1_right.shape}')
print('T2 data shape in train T1: ')
print(f'Front: {train_t2_front.shape}')
print(f'Back: {train_t2_back.shape}')
print(f'Left: {train_t2_left.shape}')
print(f'Right: {train_t2_right.shape}')

T1 data shape in test T1: 
Front: (16464, 33)
Back: (12520, 33)
Left: (14485, 33)
Right: (14485, 33)
T2 data shape in test T1: 
Front: (8756, 33)
Back: (7321, 33)
Left: (8495, 33)
Right: (8495, 33)
T1 data shape in train T1: 
Front: (73643, 69)
Back: (71040, 69)
Left: (73750, 69)
Right: (64022, 69)
T2 data shape in train T1: 
Front: (18910, 35)
Back: (17440, 35)
Left: (15082, 35)
Right: (12017, 35)
